# Fine-Tune Task SLM with Unsloth

This notebook fine-tunes Llama 3.1 8B for specific organizational tasks using Unsloth + LoRA.

**Phase 2 of Enterprise AI Habitat Project**

## Configuration

Set your task configuration below:

In [ ]:
# ========================================
# CONFIGURATION - Edit these values
# ========================================

# Unit and Task Selection
UNIT_ID = "fundraising"  # Options: fundraising, field_operations, business_development
TASK_ID = "investor_profiling"  # See task definitions in config/tasks/

# Training Settings
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 3e-4

# LoRA Settings (matches Phase 3 MoE expectations)
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Test Mode (use fewer samples and epochs)
TEST_MODE = True
TEST_SAMPLES = 50
TEST_EPOCHS = 1

# Output
SAVE_TO_HUB = False
HUB_MODEL_NAME = f"task-slm-{UNIT_ID}-{TASK_ID}"

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --progress-bar off
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton --progress-bar off
!pip install pyyaml structlog --progress-bar off

In [ ]:
import torch
from trl import SFTTrainer
from datasets import Dataset
from transformers import TrainingArguments, TextStreamer
from unsloth import FastLanguageModel, is_bfloat16_supported
import json
import random

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Task Definitions

Define system prompts and example generators for each task.

In [ ]:
# Task Definitions
TASK_DEFINITIONS = {
    "fundraising": {
        "investor_profiling": {
            "name": "Investor Profiling",
            "system_prompt": """You are an expert investment analyst specializing in angel investor profiling. Your role is to create comprehensive, actionable profiles of angel investors based on available data.

When creating investor profiles, include:
- Investment thesis and focus areas
- Historical investment patterns
- Typical investment size and stage preferences
- Industry sector preferences
- Geographic focus
- Co-investment patterns
- Portfolio company outcomes
- Known decision-making criteria

Provide structured, factual analysis that helps identify alignment with potential opportunities.""",
            "required_sections": ["Investment Thesis", "Historical Patterns", "Preferences", "Key Insights"],
        },
        "fit_assessment": {
            "name": "Investor-Opportunity Fit Assessment",
            "system_prompt": """You are an expert at assessing investor-opportunity fit. Your role is to analyze how well a specific investment opportunity aligns with an angel investor's profile, preferences, and investment thesis.

When assessing fit, evaluate:
- Thesis alignment (sector, stage, geography)
- Check size compatibility
- Value-add potential
- Portfolio synergies or conflicts
- Risk tolerance match
- Time horizon alignment

Provide a clear fit score with detailed reasoning for each dimension.""",
            "required_sections": ["Fit Score", "Alignment Analysis", "Potential Concerns", "Recommendation"],
        },
    },
    "field_operations": {
        "market_assessment": {
            "name": "Local Market Assessment",
            "system_prompt": """You are an expert at assessing local market conditions for development program implementation. Your role is to analyze market readiness, enabling environment, and implementation feasibility.

When assessing markets, evaluate:
- Regulatory environment and policy landscape
- Market infrastructure maturity
- Financial system accessibility
- Human capital availability
- Technology adoption levels
- Cultural and social factors
- Political stability indicators
- Economic conditions

Provide comprehensive market assessments with clear readiness scores and implementation recommendations.""",
            "required_sections": ["Market Overview", "Readiness Assessment", "Key Enablers", "Key Barriers", "Recommendations"],
        },
    },
    "business_development": {
        "rfp_analysis": {
            "name": "RFP Requirements Analysis",
            "system_prompt": """You are an expert at analyzing Requests for Proposals (RFPs) and funding opportunities. Your role is to extract, structure, and prioritize requirements from complex RFP documents.

When analyzing RFPs, identify:
- Mandatory requirements and eligibility criteria
- Technical requirements and specifications
- Evaluation criteria and weights
- Budget parameters and restrictions
- Timeline and milestone requirements
- Submission format and documentation
- Key decision makers and process
- Hidden requirements and preferences

Provide comprehensive requirement matrices that maximize proposal compliance and scoring potential.""",
            "required_sections": ["Eligibility Requirements", "Technical Requirements", "Evaluation Criteria", "Key Dates", "Compliance Checklist"],
        },
    },
}

# Get current task definition
task_def = TASK_DEFINITIONS.get(UNIT_ID, {}).get(TASK_ID)
if not task_def:
    raise ValueError(f"Task not found: {UNIT_ID}/{TASK_ID}")

print(f"Task: {task_def['name']}")
print(f"Required sections: {task_def['required_sections']}")

## 3. Generate Mock Training Data

For testing, we generate synthetic training data. In production, replace this with real data.

In [ ]:
def generate_mock_data(unit_id, task_id, num_samples=50):
    """Generate mock training data for a task."""
    random.seed(42)
    
    templates = {
        "fundraising": {
            "investor_profiling": {
                "inputs": [
                    "Profile investor {name} who focuses on {sector} startups",
                    "Create a comprehensive profile for {name}, an angel investor in {location}",
                    "Analyze the investment history of {name}",
                ],
                "output_template": """## Investment Thesis
{name} focuses on {sector} investments, typically at the {stage} stage.

## Historical Patterns
- Average check size: ${check_size}
- Investments per year: {investments_per_year}
- Preferred sectors: {sector}, {secondary_sector}

## Preferences
- Stage: {stage}
- Geography: {location}
- Team requirements: Technical founders preferred

## Key Insights
{name} shows strong preference for {sector} with proven traction. Co-invests frequently with established angel networks.""",
                "variables": {
                    "name": ["John Smith", "Sarah Chen", "Michael Johnson", "Emily Rodriguez", "David Kim"],
                    "sector": ["fintech", "healthtech", "cleantech", "edtech", "enterprise SaaS"],
                    "secondary_sector": ["sustainability", "digital health", "marketplaces", "developer tools"],
                    "location": ["San Francisco", "New York", "Boston", "Austin", "Seattle"],
                    "stage": ["pre-seed", "seed", "Series A"],
                    "check_size": ["25,000-50,000", "50,000-100,000", "100,000-250,000"],
                    "investments_per_year": ["3-5", "5-8", "8-12"],
                },
            },
            "fit_assessment": {
                "inputs": [
                    "Assess fit between {investor} and {company} in {sector}",
                    "Is {investor} a good match for our {stage} {sector} startup?",
                ],
                "output_template": """## Fit Score: {score}/100

## Alignment Analysis
- Sector alignment: {sector_fit}
- Stage alignment: {stage_fit}
- Check size compatibility: {check_fit}

## Potential Concerns
- {concern}

## Recommendation
{recommendation}""",
                "variables": {
                    "investor": ["John Smith", "Sarah Chen", "Michael Johnson"],
                    "company": ["TechCo", "HealthStart", "GreenFuture", "EduLearn"],
                    "sector": ["fintech", "healthtech", "cleantech"],
                    "stage": ["pre-seed", "seed", "Series A"],
                    "score": ["45", "62", "78", "85"],
                    "sector_fit": ["Strong - direct thesis match", "Moderate - adjacent interest"],
                    "stage_fit": ["Excellent - preferred stage", "Good - within range"],
                    "check_fit": ["Compatible", "Below typical range"],
                    "concern": ["Portfolio conflict", "Limited bandwidth", "Check size mismatch"],
                    "recommendation": ["Strong fit - prioritize outreach", "Good potential - worth pursuing", "Moderate fit - include in broader list"],
                },
            },
        },
        "field_operations": {
            "market_assessment": {
                "inputs": [
                    "Assess market readiness in {country}",
                    "Is {country} ready for {program_type} implementation?",
                ],
                "output_template": """## Market Overview
{country} presents {overall_assessment} conditions for {program_type} implementation.

## Readiness Assessment
- Regulatory: {regulatory}
- Infrastructure: {infrastructure}
- Human capital: {human_capital}

## Key Enablers
- {enabler1}
- {enabler2}

## Key Barriers
- {barrier1}
- {barrier2}

## Recommendations
{recommendation}""",
                "variables": {
                    "country": ["Kenya", "Nigeria", "Indonesia", "Philippines", "Colombia"],
                    "program_type": ["microfinance", "agricultural extension", "digital health"],
                    "overall_assessment": ["favorable", "moderate", "challenging"],
                    "regulatory": ["Supportive framework", "Neutral environment", "Restrictive regulations"],
                    "infrastructure": ["Strong digital infrastructure", "Limited connectivity", "Improving rapidly"],
                    "human_capital": ["Skilled workforce available", "Training needs identified"],
                    "enabler1": ["Mobile money penetration", "Government support", "Strong NGO ecosystem"],
                    "enabler2": ["Youth population", "Growing middle class", "Technology adoption"],
                    "barrier1": ["Political instability", "Currency volatility", "Infrastructure gaps"],
                    "barrier2": ["Limited local partners", "Regulatory uncertainty", "Competition for talent"],
                    "recommendation": ["Proceed with pilot program", "Delay until regulatory clarity", "Partner with established local organization"],
                },
            },
        },
        "business_development": {
            "rfp_analysis": {
                "inputs": [
                    "Analyze RFP from {funder} for {program_type}",
                    "What are the requirements for {funder}'s RFP?",
                ],
                "output_template": """## Eligibility Requirements
- Organization type: {org_type}
- Geographic presence: {geography}
- Prior experience: {experience}

## Technical Requirements
- {tech_req1}
- {tech_req2}

## Evaluation Criteria
| Criterion | Weight |
|-----------|--------|
| Technical approach | {weight1}% |
| Past performance | {weight2}% |
| Cost | {weight3}% |

## Key Dates
- Questions due: {questions_date}
- Submission deadline: {deadline}

## Compliance Checklist
- [ ] {checklist1}
- [ ] {checklist2}""",
                "variables": {
                    "funder": ["USAID", "World Bank", "Gates Foundation", "DFID"],
                    "program_type": ["health systems", "agricultural development", "education"],
                    "org_type": ["501(c)(3) or equivalent", "International NGO"],
                    "geography": ["Must have presence in target countries", "Headquarters in DAC country"],
                    "experience": ["5+ years in sector", "Prior USAID experience preferred"],
                    "tech_req1": ["Evidence-based methodology", "M&E framework required"],
                    "tech_req2": ["Local partner involvement", "Sustainability plan"],
                    "weight1": ["40", "50", "35"],
                    "weight2": ["30", "25", "35"],
                    "weight3": ["30", "25", "30"],
                    "questions_date": ["January 15", "February 1", "March 10"],
                    "deadline": ["February 28", "March 15", "April 30"],
                    "checklist1": ["SAM.gov registration", "DUNS number"],
                    "checklist2": ["Past performance references", "Key personnel CVs"],
                },
            },
        },
    }
    
    template = templates.get(unit_id, {}).get(task_id)
    if not template:
        raise ValueError(f"No template for {unit_id}/{task_id}")
    
    examples = []
    for i in range(num_samples):
        # Select random values for variables
        vars_instance = {k: random.choice(v) for k, v in template["variables"].items()}
        
        # Format input and output
        input_text = random.choice(template["inputs"]).format(**vars_instance)
        output_text = template["output_template"].format(**vars_instance)
        
        examples.append({"input": input_text, "output": output_text})
    
    return examples

# Generate data
num_samples = TEST_SAMPLES if TEST_MODE else 300
examples = generate_mock_data(UNIT_ID, TASK_ID, num_samples)
print(f"Generated {len(examples)} training examples")
print(f"\nSample input: {examples[0]['input']}")
print(f"\nSample output preview: {examples[0]['output'][:300]}...")

## 4. Load Model

In [ ]:
# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
)

print(model.print_trainable_parameters())

## 5. Format Training Data

In [ ]:
# ChatML format for Llama 3.1
def format_chatml(system_prompt, user_message, assistant_response):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{assistant_response}<|eot_id|>"""

# Format examples
system_prompt = task_def["system_prompt"]
formatted_data = [
    {"text": format_chatml(system_prompt, ex["input"], ex["output"])}
    for ex in examples
]

# Create dataset
train_dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(train_dataset)}")
print(f"\nSample formatted text:\n{train_dataset[0]['text'][:500]}...")

## 6. Train Model

In [ ]:
# Training configuration
epochs = TEST_EPOCHS if TEST_MODE else NUM_EPOCHS
batch_size = 2 if TEST_MODE else BATCH_SIZE

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=epochs,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir=f"outputs/{UNIT_ID}/{TASK_ID}",
        seed=42,
        save_strategy="epoch",
    ),
)

print(f"Training {UNIT_ID}/{TASK_ID} for {epochs} epoch(s)...")
trainer.train()

## 7. Test Inference

In [ ]:
# Prepare for inference
model = FastLanguageModel.for_inference(model)

# Test prompt
test_input = examples[0]["input"]
test_prompt = format_chatml(system_prompt, test_input, "")

# Generate
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

print(f"Input: {test_input}\n")
print("Generated output:")
print("-" * 50)

text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
)

## 8. Save Model

In [ ]:
# Save adapter only (recommended for Phase 3 MoE)
output_path = f"models/{UNIT_ID}/{TASK_ID}_v1"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

# Save training metadata
metadata = {
    "unit_id": UNIT_ID,
    "task_id": TASK_ID,
    "task_name": task_def["name"],
    "base_model": "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "lora_config": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    },
    "training_config": {
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": LEARNING_RATE,
        "train_samples": len(examples),
    },
    "required_sections": task_def["required_sections"],
}

with open(f"{output_path}/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to {output_path}")

In [ ]:
# Optional: Save merged model (larger file)
if False:  # Set to True to save merged model
    merged_path = f"models/{UNIT_ID}/{TASK_ID}_v1_merged"
    model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to {merged_path}")

In [ ]:
# Optional: Push to HuggingFace Hub
if SAVE_TO_HUB:
    from huggingface_hub import login
    login()  # Enter your HF token
    
    model.push_to_hub(HUB_MODEL_NAME)
    tokenizer.push_to_hub(HUB_MODEL_NAME)
    print(f"Model pushed to {HUB_MODEL_NAME}")

## 9. Download Model

Download the trained adapter to your local machine.

In [ ]:
# Zip and download
import shutil
shutil.make_archive(f"{UNIT_ID}_{TASK_ID}_adapter", 'zip', output_path)

from google.colab import files
files.download(f"{UNIT_ID}_{TASK_ID}_adapter.zip")